# AI Code Reviewer and fixer

## imports

In [4]:
!pip install -q langchain langchain-community langchain-groq gradio requests

In [14]:
import os
import gradio as gr
import requests
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate


In [ ]:
os.environ["GROQ_API_KEY"] = "grok_key"

## LLM class

In [20]:

class OpenSourceLLM(ChatGroq):
    def __init__(self):
        print("🤖 Agentic AI loaded — ready to plan and code")
        
        # Get API key from environment (secure way)
        api_key = os.getenv("GROQ_API_KEY")
        
        if not api_key:
            raise ValueError(
                "❌ GROQ_API_KEY not found!\n"
                "   → In Kaggle: Add it via Add-ons → Secrets (name: GROQ_API_KEY)\n"
                "   → Locally: export GROQ_API_KEY='your_key'\n"
                "   → Never hardcode keys in public code!"
            )
        
        super().__init__(
            model="llama-3.3-70b-versatile",
            temperature=0.7,
            max_tokens=1000,
            groq_api_key=api_key  # Correct parameter name
        )

# Create instance
llm = OpenSourceLLM()

🤖 Agentic AI loaded — ready to plan and code


## Tools

In [11]:

@tool
def run_code(code: str) -> str:
    """Safely execute Python code and return output or error."""
    try:
        local_vars = {}
        exec(code, {"__builtins__": {}}, local_vars)  
        return str(local_vars.get('result', "Code ran but no 'result' variable set"))
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def web_search(query: str) -> str:
    """Search the web for coding help."""
    return "Web search results for '{query}': [simulated info] Use Google or StackOverflow for details."

tools = [run_code, web_search]

## Agent definition andsettings

In [22]:

agent_prompt = PromptTemplate.from_template("""
You are an AI coding agent. Answer the user's request using the tools available.

TOOLS:
------
You have access to the following tools:

{tools}

To use a tool, please use the following format:

Thought: Do I need to use a tool? Yes
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action

When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:

Thought: Do I need to use a tool? No
Final Answer: [your response here]

Begin!

Question: {input}
Thought: {agent_scratchpad}
""")

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=agent_prompt
)

memory = ConversationBufferMemory(memory_key="chat_history")

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True,  
    early_stopping_method="generate"
)

## Truyal

In [23]:
response = agent_executor.invoke({"input": "Build a simple Python function to calculate factorial"})
print(response["output"])



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? No
Final Answer: Here's a simple Python function to calculate the factorial of a number:

```python
def calculate_factorial(n):
    if n == 0:
        return 1
    else:
        return n * calculate_factorial(n-1)
```

You can use this function by calling it with a non-negative integer, like this: `calculate_factorial(5)`. This would return `120`, which is the factorial of 5. 

Alternatively, you could use an iterative approach:

```python
def calculate_factorial(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result
```

Both of these functions will give you the correct result, but the iterative version might be slightly more efficient for large inputs.

> Finished chain.
Here's a simple Python function to calculate the factorial of a number:

```python
def calculate_factorial(n):
    if n == 0:
        return 1
    else:
        return n * calculate_factorial(n-1)
```

You can us

## Gradio interface

In [25]:
def agent_chat(user_input):
    response = agent_executor.invoke({"input": user_input})
    return response["output"]

with gr.Blocks() as demo:
    gr.Markdown("# Self-Improving Agentic AI Code Builder")
    input_box = gr.Textbox(placeholder="Task: e.g., 'Build a Flask todo API'")
    output = gr.Markdown(label="Agent Response")
    submit = gr.Button("Run Agent")

    submit.click(agent_chat, inputs=input_box, outputs=output)

demo.queue()
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://256ae17116463cfb62.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
